In [40]:
import pickle
import pandas as pd

from tensorflow.keras.models import load_model


### Load all the model pickle,h5

In [41]:
model = load_model("model.h5")

with open("label_encoder_gender.pkl","rb") as file:
    label_encoder_gender = pickle.load(file)

with open("one_hot_encoder_geography.pkl","rb") as file:
    one_hot_encoder_geography = pickle.load(file)

with open("standard_scaler.pkl","rb") as file:
    standard_scaler = pickle.load(file)

# example input data

In [42]:
input_data = {
    "CreditScore" :600,
    "Geography" : "France",
    "Gender" :"Male",
    "Age" : 30,
    "Tenure": 3,
    "Balance" : 60000,
    "NumOfProducts":2,
    "HasCrCard":1,
    "IsActiveMember":1,
    "EstimatedSalary":50000
}

In [43]:
input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,30,3,60000,2,1,1,50000


### gender encoding

In [44]:
input_data_df["Gender"] = label_encoder_gender.transform(input_data_df["Gender"])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,30,3,60000,2,1,1,50000


### geography markdown

In [45]:
geo_encoded =  one_hot_encoder_geography.transform([input_data_df["Geography"]]).toarray()
geo_encoded

d:\PROJECTS\ANN_project_deployment\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([[1., 0., 0.]])

In [46]:
geo_encoded_df =  pd.DataFrame(geo_encoded,columns =one_hot_encoder_geography.get_feature_names_out(["Geography"]))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [51]:
input_data_final= pd.concat([input_data_df.reset_index(drop=True),geo_encoded_df],axis =1)
input_data_final = input_data_final.drop("Geography", axis =1)
input_data_final

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,30,3,60000,2,1,1,50000,1.0,0.0,0.0


## Scaling

In [52]:
input_data_scaled = standard_scaler.transform(input_data_final)
input_data_scaled

array([[-0.53598516,  0.91324755, -0.84593077, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

# Prediction Churn

In [53]:
prediction = model.predict(input_data_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step


array([[0.01722972]], dtype=float32)

In [54]:
prediction_probablity = prediction[0][0]
prediction_probablity

np.float32(0.017229725)

In [56]:
if prediction_probablity > 0.5:
    print("not likely to churn")
else:
    print("likely to churn")

likely to churn
